In [5]:
import pandas as pd
import plotly.graph_objects as go

def plot_prosperity_day(day_str, product='HYDROGEL_PACK'):
    """
    Loads data for a specific day and plots the interactive line chart
    with EMAs and all 3 levels of order book depth.
    """
    # 1. Load the data for the requested day (Updated for Round 3)
    df_prices = pd.read_csv(f'dataset/prices_round_3_day_{day_str}.csv', sep=';') 
    df_trades = pd.read_csv(f'dataset/trades_round_3_day_{day_str}.csv', sep=';')

    # Filter datasets for the chosen product
    p_df = df_prices[df_prices['product'] == product].copy()
    t_df = df_trades[df_trades['symbol'] == product].copy()

    # Filter out invalid ticks where the order book clears out
    p_df = p_df[p_df['mid_price'] > 0]

    # 2. Calculate EMAs directly on the mid_price
    p_df['EMA_5'] = p_df['mid_price'].ewm(span=5, adjust=False).mean()
    p_df['EMA_20'] = p_df['mid_price'].ewm(span=20, adjust=False).mean()
    p_df['EMA_50'] = p_df['mid_price'].ewm(span=50, adjust=False).mean()

    fig = go.Figure()

    # -------------------------------------------------------------
    # 3. ADD ORDER BOOK DEPTH (Plotted first to stay in background)
    # -------------------------------------------------------------
    
    # BIDS - Fading blue lines for depth
    bid_colors = ['rgba(50, 100, 250, 1)', 'rgba(50, 100, 250, 0.75)', 'rgba(50, 100, 250, 0.5)']
    for i in range(1, 4):
        col_name = f'bid_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=bid_colors[i-1], width=1 if i==1 else 0.5),
                name=f'Bid L{i}',
                hoverinfo='skip' 
            ))

    # ASKS - Fading red lines for depth
    ask_colors = ['rgba(250, 50, 50, 1)', 'rgba(250, 50, 50, 0.75)', 'rgba(250, 50, 50, 0.5)']
    for i in range(1, 4):
        col_name = f'ask_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=ask_colors[i-1], width=1 if i==1 else 0.5),
                name=f'Ask L{i}',
                hoverinfo='skip'
            ))

    # -------------------------------------------------------------
    # 4. ADD FOREGROUND ELEMENTS (Mid Price Line, EMAs, & Trades)
    # -------------------------------------------------------------
    
    # Add Mid Price Line
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'],
        y=p_df['mid_price'],
        mode='lines',
        line=dict(color='#ffffff', width=2),
        name='Mid Price'
    ))

    # Add EMAs
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'], y=p_df['EMA_5'], mode='lines',
        line=dict(color='#00e676', width=1.5), name='EMA 5' 
    ))
    
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'], y=p_df['EMA_20'], mode='lines',
        line=dict(color='#ffea00', width=1.5), name='EMA 20' 
    ))
    
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'], y=p_df['EMA_50'], mode='lines',
        line=dict(color='#d500f9', width=1.5), name='EMA 50' 
    ))

    # Add Trade Markers
    if not t_df.empty:
        fig.add_trace(go.Scatter(
            x=t_df['timestamp'],
            y=t_df['price'],
            mode='markers',
            marker=dict(color='#ff9800', size=8, symbol='circle', line=dict(color='white', width=1)),
            name='Trades',
            customdata=t_df['quantity'],
            hovertemplate='Price: %{y}<br>Volume: %{customdata}<extra></extra>'
        ))

    # 5. Format Layout
    fig.update_layout(
        title=f'{product} - Day {day_str} (Mid Price & EMAs) w/ Order Depth',
        xaxis_title='Timestamp',
        yaxis_title='Price (XIRECs)',
        template='plotly_dark', 
        xaxis_rangeslider_visible=True,
        yaxis=dict(fixedrange=False), 
        hovermode='x unified', 
        height=700
    )

    # Display the interactive chart in the browser
    fig.show(renderer="browser")

# --- RUN THE VISUALIZATIONS ---
# Restricted to core products + 1 ATM option to prevent browser crash
products_to_plot = ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]
for i in [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500]:
    products_to_plot.append(f"VEV_{i}")

for prd in products_to_plot:
    for i in ["0", "1", "2"]:
        plot_prosperity_day(i, prd)

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

# --- 1. VECTORIZED BLACK-SCHOLES GREEKS ---
def calculate_greeks(S, K, T, r, sigma):
    """
    Vectorized calculation of Call Option Greeks.
    S, K, T can be Pandas Series or NumPy arrays.
    """
    # Prevent division by zero as T approaches 0
    T = np.maximum(T, 1e-8)
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # PDF and CDF of Standard Normal
    N_d1 = norm.cdf(d1)
    N_d2 = norm.cdf(d2)
    pdf_d1 = norm.pdf(d1)
    
    # Greeks
    delta = N_d1
    gamma = pdf_d1 / (S * sigma * np.sqrt(T))
    # Theta (Annualized)
    theta = -(S * pdf_d1 * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * N_d2
    vega = S * np.sqrt(T) * pdf_d1
    
    return delta, gamma, theta, vega

# --- 2. DATA PIPELINE ---
def load_and_prep_data(days=[0, 1, 2]):
    print("Loading and aligning datasets...")
    frames = []
    
    for day in days:
        df = pd.read_csv(f'prices_round_3_day_{day}.csv', sep=';')
        df['day'] = day
        # Create a continuous global timestamp (each day has 1,000,000 ticks)
        df['global_time'] = df['timestamp'] + (day * 1000000)
        frames.append(df)
        
    full_df = pd.concat(frames, ignore_index=True)
    full_df = full_df[full_df['mid_price'] > 0] # Clean dead ticks
    
    # Pivot so every timestamp has the underlying and all options side-by-side
    pivot_df = full_df.pivot(index='global_time', columns='product', values='mid_price')
    pivot_df['day'] = full_df.groupby('global_time')['day'].first()
    pivot_df['timestamp'] = full_df.groupby('global_time')['timestamp'].first()
    
    # Forward fill missing ticks (if any)
    pivot_df.fillna(method='ffill', inplace=True)
    pivot_df.dropna(inplace=True)
    
    return pivot_df

# --- 3. EXECUTION ---
def run_quant_analysis():
    df = load_and_prep_data([0, 1, 2])
    
    underlying = 'VELVETFRUIT_EXTRACT'
    strikes = [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500]
    options = [f'VEV_{s}' for s in strikes]
    
    # Hardcoded simulation constants
    r = 0.0
    sigma = 0.252
    
    print("Calculating Vectorized Greeks for all ticks...")
    # Calculate Time to Expiry (T). Starts at 5 days, decreases continuously.
    # T = (5 - day - (timestamp / 1,000,000)) / 252
    df['T_years'] = (5.0 - df['day'] - (df['timestamp'] / 1000000.0)) / 252.0
    
    S = df[underlying].values
    T = df['T_years'].values
    
    for K in strikes:
        opt = f'VEV_{K}'
        delta, gamma, theta, vega = calculate_greeks(S, K, T, r, sigma)
        
        df[f'{opt}_Delta'] = delta
        df[f'{opt}_Gamma'] = gamma
        df[f'{opt}_Theta'] = theta
        df[f'{opt}_Vega'] = vega

    # --- 4. CORRELATION ANALYSIS ---
    print("\nMapping Correlation Matrices...")
    # Calculate price return correlations (more statistically sound than absolute price)
    returns_df = df[[underlying] + options].pct_change().dropna()
    corr_matrix = returns_df.corr()
    
    # Display Underlying vs Option Correlations
    print("\n--- Correlation vs Underlying (VELVETFRUIT_EXTRACT) ---")
    underlying_corrs = corr_matrix[underlying].sort_values(ascending=False)
    print(underlying_corrs)

    # --- 5. VISUALIZATION ---
    plt.style.use('dark_background')
    
    # Plot 1: Gamma Profile over Time for ATM Options
    plt.figure(figsize=(12, 6))
    atm_strikes = [5100, 5200, 5300]
    for K in atm_strikes:
        plt.plot(df.index, df[f'VEV_{K}_Gamma'], label=f'Strike {K} Gamma', alpha=0.8)
    
    plt.title('Gamma Explosion as Time to Expiry Decreases (Days 0 -> 2)')
    plt.xlabel('Global Timestamp')
    plt.ylabel('Gamma (ΔDelta / ΔPrice)')
    plt.axvline(1000000, color='grey', linestyle='--', label='Start Day 1')
    plt.axvline(2000000, color='grey', linestyle='--', label='Start Day 2')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.savefig('gamma_explosion.png')
    plt.close()
    
    # Plot 2: Correlation Heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=0, vmax=1)
    plt.title('Return Correlation Matrix (Options vs Underlying)')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png')
    plt.close()
    
    print("\nAnalysis Complete. Visualizations saved as 'gamma_explosion.png' and 'correlation_heatmap.png'.")

if __name__ == "__main__":
    run_quant_analysis()

Loading and aligning datasets...


FileNotFoundError: [Errno 2] No such file or directory: 'prices_round_3_day_0.csv'